# Phase-3D campaign progress

Live status over `results/phase3d/hy{hy}/{electric_hx{hx}|magnetic_hz{hz}}/L{L}/` --
coverage of the planned grid, health of what has landed, learning curves, the
order-parameter sweeps landed so far, and a running read of the transition location
per (cut, L). Everything below degrades to an empty table/plot on a partial or empty
campaign -- rerun any time, it never assumes the run is finished.

Locator fits reuse `analysis/scripts/transition_fit.py` (untracked peer module).


## 1. Config

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

# repo root, found from wherever nbconvert's cwd lands (project convention: notebooks
# run with cwd = analysis/notebooks/, but this works regardless)
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "analysis" / "scripts"))
import phase3d_status as ps           # noqa: E402
import transition_fit as tf           # noqa: E402  (untracked peer module; import-only)

# --- knobs --------------------------------------------------------------------------
# Env-var overridable so the SAME committed config runs on today's empty results/phase3d
# and on a stand-in tree (set PHASE3D_CAMPAIGN / PHASE3D_DATA_CURVES for a dry run).
CAMPAIGN = Path(os.environ.get("PHASE3D_CAMPAIGN", ROOT / "results" / "phase3d"))
DATA_CURVES = Path(os.environ.get("PHASE3D_DATA_CURVES", ROOT / "data" / "tc_nqs" / "phase3d"))
FIGS = ROOT / "analysis" / "figs"
HY_LIST = None        # None = auto-detect from what's landed/planned; else e.g. [0.0, 0.2, 0.4]
L_LIST = None         # None = auto-detect; else e.g. [4, 5, 6]
MIN_POINTS = 5        # locator gate: >= this many landed points on a cut before fitting h_c(L)
WRITE_STATUS = True   # section 7: write CAMPAIGN/STATUS.md

print(f"ROOT        = {ROOT}")
print(f"CAMPAIGN    = {CAMPAIGN}  (exists: {CAMPAIGN.exists()})")
print(f"DATA_CURVES = {DATA_CURVES}  (exists: {DATA_CURVES.exists()})")


In [ ]:
def show_scrollable(df, max_height=360):
    """House style: wide tables get their own scrollable container."""
    if df is None or df.empty:
        display(HTML('<i>(no rows)</i>'))
        return
    html = df.to_html(index=False, na_rep="--", float_format=lambda x: f"{x:.4g}")
    display(HTML(f'<div style="max-width:100%; overflow-x:auto; max-height:{max_height}px; '
                 f'overflow-y:auto; border:1px solid #ddd; padding:4px;">{html}</div>'))


# --- load everything once ------------------------------------------------------------
finals = ps.add_health(ps.load_finals(CAMPAIGN))
manifests = ps.load_manifests(CAMPAIGN)
watch = ps.load_watch_state(CAMPAIGN)
planned = ps.load_planned(CAMPAIGN)
coverage = ps.coverage(planned, finals)
locators = ps.partial_locators(CAMPAIGN, finals, min_points=MIN_POINTS)

if HY_LIST is None:
    HY_LIST = sorted((set(finals["hy"]) if len(finals) else set())
                      | (set(coverage["hy"]) if len(coverage) else set()))
if L_LIST is None:
    L_LIST = sorted((set(finals["L"]) if len(finals) else set())
                     | (set(coverage["L"]) if len(coverage) else set()))
CUTS = ["electric", "magnetic"]

print(f"{len(finals)} landed runs, {len(manifests)} manifest rows, {len(watch)} watch_state entries")
print(f"hy in {HY_LIST}")
print(f"L  in {L_LIST}")


## 2. Coverage heat-grid
Rows = cuts, columns = L, one panel per h_y. Cell text is landed/planned; colour is the
landed fraction; a red hatch marks a cell with at least one diverged run.


In [ ]:
def plot_coverage(hy_list, l_list, cov):
    if not hy_list or not l_list:
        print("No coverage data yet (empty campaign).")
        return
    fig, axes = plt.subplots(1, len(hy_list), figsize=(3.4 * len(hy_list), 2.8), squeeze=False)
    axes = axes[0]
    for ax, hy in zip(axes, hy_list):
        sub = cov[np.isclose(cov["hy"], hy)]
        grid = np.full((len(CUTS), len(l_list)), np.nan)
        for i, cut in enumerate(CUTS):
            for j, L in enumerate(l_list):
                row = sub[(sub.cut == cut) & (sub.L == L)]
                if len(row):
                    grid[i, j] = row.iloc[0]["frac"]
        ax.imshow(grid, vmin=0, vmax=1, cmap="plasma", aspect="auto")
        for i, cut in enumerate(CUTS):
            for j, L in enumerate(l_list):
                row = sub[(sub.cut == cut) & (sub.L == L)]
                if len(row):
                    r = row.iloc[0]
                    ax.text(j, i, f"{int(r['landed'])}/{int(r['planned'])}", ha="center", va="center",
                            fontsize=8, color="white" if r["frac"] > 0.5 else "black")
                    if r["diverged"] > 0:
                        ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False,
                                                    hatch="////", edgecolor="crimson", linewidth=0))
                else:
                    ax.text(j, i, "--", ha="center", va="center", fontsize=8, color="gray")
        ax.set_xticks(range(len(l_list)), [f"L={L}" for L in l_list])
        ax.set_yticks(range(len(CUTS)), CUTS)
        ax.set_title(f"h_y = {hy:g}")
        ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
    fig.suptitle("Coverage: landed / planned  (hatched = has diverged runs)")
    fig.tight_layout()
    # plt.savefig(FIGS / "phase3d_coverage.png", dpi=300, bbox_inches="tight")
    plt.show()


plot_coverage(HY_LIST, L_LIST, coverage)


## 3. Job / health table
Every submitted job (manifest) joined with its live state (watch_state) and, once
landed, its final observables -- state, last step, diverged, warm-started, E0 vs the
h=0 bound, Vscore.


In [ ]:
job_tbl = ps.job_table(CAMPAIGN, df=finals, manifests=manifests, watch=watch)
if job_tbl.empty:
    print("No jobs recorded yet (no manifests or finals under CAMPAIGN).")
else:
    show_scrollable(job_tbl)


## 4. Learning curves
E vs step, coloured by L, for the (h_y, cut) plane with the most L coverage so far, at
the field point shared by the most sizes. Dashed = the h=0 OBC bound for that L; dotted
= `ref_E` where the run carries one. Reads `DATA_CURVES/.../<name>.curve.json` first,
falling back to the run's own inline `curve` field (both schemas land in the wild).


In [ ]:
def load_curve(row):
    try:
        rel = Path(row["path"]).relative_to(CAMPAIGN)
    except ValueError:
        rel = None
    if rel is not None:
        side = DATA_CURVES / rel.parent / f"{row['name']}.curve.json"
        if side.exists():
            try:
                j = json.loads(side.read_text())
                if "curve" in j:
                    return j["curve"]
            except (json.JSONDecodeError, OSError):
                pass
    try:
        return json.loads(Path(row["path"]).read_text()).get("curve")
    except (json.JSONDecodeError, OSError):
        return None


def pick_plane(df):
    if df.empty:
        return None, None, None
    counts = df.groupby(["hy", "cut"])["L"].nunique().sort_values(ascending=False)
    hy, cut = counts.index[0]
    plane = df[(df.hy == hy) & (df.cut == cut) & (~df.diverged)]
    h_counts = plane.groupby("h")["L"].nunique().sort_values(ascending=False)
    h = float(h_counts.index[0]) if len(h_counts) else None
    return hy, cut, h


sel_hy, sel_cut, sel_h = pick_plane(finals)
if sel_hy is None:
    print("No landed runs yet -- nothing to plot.")
else:
    sub = finals[(finals.hy == sel_hy) & (finals.cut == sel_cut) & np.isclose(finals.h, sel_h)]
    Ls_here = sorted(sub.L.unique())
    colors = tf.plasma_by_L(Ls_here)
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    any_curve = False
    for _, row in sub.sort_values("L").iterrows():
        cv = load_curve(row)
        if not cv or not cv.get("step"):
            continue
        any_curve = True
        c = colors[row["L"]]
        ax.plot(cv["step"], cv["energy"], color=c, lw=1.2, label=f"L={row['L']}")
        ax.axhline(ps.bound(row["L"]), color=c, ls="--", lw=1, alpha=0.6)
        if pd.notna(row.get("ref_E")):
            ax.axhline(row["ref_E"], color=c, ls=":", lw=1, alpha=0.8)
    if any_curve:
        ax.set_xlabel("step")
        ax.set_ylabel("E")
        ax.set_title(f"h_y={sel_hy:g}, {sel_cut} cut, h={sel_h:g}  (dashed=h=0 bound, dotted=ref_E)")
        ax.legend(frameon=False)
        tf.openax(ax)
        # plt.savefig(FIGS / "phase3d_learning_curves.png", dpi=300, bbox_inches="tight")
        plt.show()
    else:
        print("Selected point has no curve data (neither a sibling .curve.json nor an inline 'curve').")


## 5. Order-parameter / observable sweeps (landed so far)
Electric cuts: `O_FM_paratoric` vs h_z, error bars magnified x3 for visibility.
Magnetic (chain) cuts: E, `sx_mean`, `A_v_mean` vs h_x per branch (up solid / dn dashed)
-- the hysteresis view.


In [ ]:
electric = finals[finals.cut == "electric"]
if electric.empty:
    print("No electric-cut data landed yet.")
else:
    for (hy, ffield, fval), g in electric.groupby(["hy", "fixed_field", "fixed_val"]):
        colors = tf.plasma_by_L(sorted(g.L.unique()))
        fig, ax = plt.subplots(figsize=(6, 4))
        for L, gl in g.groupby("L"):
            gl = gl.sort_values("h")
            ax.errorbar(gl.h, gl.O_FM_paratoric, yerr=3 * gl.O_FM_paratoric_err, color=colors[L],
                        marker="o", ms=5, capsize=2, lw=1, label=f"L={L} (bars ×3)")
        ax.set_xlabel(f"h_z  (h_y={hy:g}, {ffield}={fval:g} fixed)")
        ax.set_ylabel("O_FM_paratoric")
        ax.set_title(f"Electric cut: h_y={hy:g}, {ffield}={fval:g}")
        ax.legend(frameon=False)
        tf.openax(ax)
        # plt.savefig(FIGS / f"phase3d_efm_hy{hy:g}_{ffield}{fval:g}.png", dpi=300, bbox_inches="tight")
        plt.show()


In [ ]:
magnetic = finals[finals.cut == "magnetic"]
if magnetic.empty:
    print("No magnetic (chain) cut data landed yet.")
else:
    for (hy, ffield, fval, L), g in magnetic.groupby(["hy", "fixed_field", "fixed_val", "L"]):
        fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
        any_branch = False
        for branch, ls in (("up", "-"), ("dn", "--")):
            gb = g[g.branch == branch].sort_values("h")
            if gb.empty:
                continue
            any_branch = True
            axes[0].plot(gb.h, gb.E0, ls, marker="o", ms=5, label=branch)
            axes[1].plot(gb.h, gb.sx, ls, marker="o", ms=5, label=branch)
            axes[2].plot(gb.h, gb.A_v, ls, marker="o", ms=5, label=branch)
        if not any_branch:
            plt.close(fig)
            continue
        for ax, yl in zip(axes, ("E0", "sx_mean", "A_v_mean")):
            ax.set_xlabel(f"h_x")
            ax.set_ylabel(yl)
            tf.openax(ax)
        axes[0].legend(frameon=False)
        fig.suptitle(f"Magnetic (chain) cut: h_y={hy:g}, {ffield}={fval:g}, L={L} -- hysteresis view")
        fig.tight_layout()
        # plt.savefig(FIGS / f"phase3d_chain_hy{hy:g}_{ffield}{fval:g}_L{L}.png", dpi=300, bbox_inches="tight")
        plt.show()


## 6. Partial locators
Electric cuts: per (h_y, L) with >= `MIN_POINTS` landed points, h_c(L) from
`transition_fit` (richards-inflection central value, the production marker policy).
Magnetic (chain) cuts: per branch, the last landed field and whether the up/dn energy
branches have crossed. Exact thermodynamic anchors and the peer's banked off-axis FSS
values (h_y=0, richards marker policy) are shown for reference -- they sit at different
(h_x, h_z) points than most campaign cuts, so agreement is not expected point-for-point.


In [ ]:
reference = pd.DataFrame([
    {"quantity": "hz_c(hx=0, hy=0)", "value": tf.EXACT["hz_c(hx=0,hy=0)"], "err": None,
     "source": "exact, (3+1)D Ising* duality (2nd order)"},
    {"quantity": "hx_c(hz=0, hy=0)", "value": tf.EXACT["hx_c(hz=0,hy=0)"], "err": None,
     "source": "exact, self-duality to 4D Wegner Z2 (1st order)"},
    {"quantity": "hz_c(hx=0.2, hy=0)", "value": 0.196, "err": 0.038,
     "source": "peer banked FSS, richards marker (off-axis electric cut)"},
    {"quantity": "hx_c(hz=0.1, hy=0)", "value": 0.98, "err": 0.19,
     "source": "peer banked FSS, richards marker (off-axis magnetic cut)"},
])
print("Reference anchors:")
show_scrollable(reference)

print("Electric-cut locators (partial FSS -- more L unlocks the true h_c(L->inf) fit):")
show_scrollable(locators["electric"])

print("Magnetic (chain) cut summary:")
show_scrollable(locators["chain"])


## 7. Write STATUS.md

In [ ]:
if WRITE_STATUS:
    out_path = CAMPAIGN / "STATUS.md"
    status_text = ps.write_status_md(CAMPAIGN, out_path, min_points=MIN_POINTS)
    print(f"wrote {out_path}")
    print(status_text)
else:
    print("WRITE_STATUS is False -- skipping STATUS.md.")
